# Exploratory Data Analysis (EDA) de Audios
Este notebook realiza un análisis exploratorio detallado sobre la base de datos de características de audios.
El objetivo principal es entender las distribuciones de las características (features) extraídas y, lo que es más importante, comparar las diferencias entre audios reales (**bonafide**) y audios generados por IA (**spoof**).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos
Vamos a cargar el archivo `mis_features_fusionadas.csv` que contiene las métricas calculadas sobre los audios.

In [2]:
df = pd.read_csv('mis_features_fusionadas.csv')
df.head()

,file_name,freq_mean,freq_std,freq_maxv,freq_minv,freq_median,freq_skew,freq_kurt,freq_q1,freq_q3,...,mfcc_mean,tempogram_mean,spec_centroid_mean,spec_bandwidth_mean,spec_contrast_mean,spec_flatness_mean,spec_rolloff_mean,User_ID,Spoofing_ID,Key
0,LA_T_1002656,-1.255777e-05,0.288675,0.499975,-0.500000,-0.000013,3.560784e-16,-1.2,-0.250006,0.249981,...,-20.945473,0.057565,1289.578927,1208.357039,35.717037,0.000589,2350.155874,LA_0085,A01,spoof
1,LA_T_1025067,-2.715178e-06,0.288675,0.499995,-0.500000,-0.000003,2.053054e-16,-1.2,-0.250001,0.249996,...,-18.226830,0.127620,1590.652731,1373.878320,35.272513,0.001294,2936.082153,LA_0083,A03,spoof
2,LA_T_1025525,-1.435542e-17,0.288675,0.499996,-0.499996,0.000000,-7.459294e-17,-1.2,-0.249998,0.249998,...,-28.607279,0.118782,2308.117893,1657.989573,31.393162,0.000620,4103.247366,LA_0079,A02,spoof
3,LA_T_1033664,-5.527916e-06,0.288675,0.499989,-0.500000,-0.000006,1.044969e-16,-1.2,-0.250003,0.249992,...,-26.618511,0.084083,1720.885540,1632.130844,34.173704,0.000539,3375.785802,LA_0088,A02,spoof
4,LA_T_1037774,0.000000e+00,0.288675,0.499995,-0.499995,0.000000,1.023792e-16,-1.2,-0.249997,0.249997,...,-16.610666,0.155019,1947.834835,1591.986941,35.753341,0.000241,3320.277160,LA_0094,A05,spoof


## 2. Información General del Dataset

In [3]:
print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.info())

El dataset tiene 725 filas y 26 columnas.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 725 entries, 0 to 724
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   file_name            725 non-null    object 
 1   freq_mean            725 non-null    float64
 2   freq_std             725 non-null    float64
 3   freq_maxv            725 non-null    float64
 4   freq_minv            725 non-null    float64
 5   freq_median          725 non-null    float64
 6   freq_skew            725 non-null    float64
 7   freq_kurt            725 non-null    float64
 8   freq_q1              725 non-null    float64
 9   freq_q3              725 non-null    float64
 10  freq_mode            725 non-null    float64
 11  freq_iqr             725 non-null    float64
 12  energy               725 non-null    float64
 13  rmse_mean            725 non-null    float64
 14  zero_crossings       725 non-null    int64  
 15

None

In [4]:
# Estadísticas descriptivas generales
df.describe().T

,count,mean,std,min,25%,50%,75%,max
freq_mean,725.0,-3.895150e-06,4.583716e-06,-2.992578e-05,-7.091193e-06,-0.000003,0.000000e+00,4.110014e-17
freq_std,725.0,2.886751e-01,5.164304e-11,2.886751e-01,2.886751e-01,0.288675,2.886751e-01,2.886751e-01
freq_maxv,725.0,4.999883e-01,6.795949e-06,4.999401e-01,4.999852e-01,0.499990,4.999932e-01,4.999979e-01
freq_minv,725.0,-4.999961e-01,4.843362e-06,-5.000000e-01,-5.000000e-01,-0.500000,-4.999930e-01,-4.999715e-01
freq_median,725.0,-3.895150e-06,4.583716e-06,-2.992578e-05,-7.091193e-06,-0.000003,0.000000e+00,0.000000e+00
freq_skew,725.0,-3.617624e-19,1.808929e-16,-5.962172e-16,-9.667328e-17,0.000000,1.068332e-16,7.259645e-16
freq_kurt,725.0,-1.200000e+00,8.587043e-10,-1.200000e+00,-1.200000e+00,-1.200000,-1.200000e+00,-1.200000e+00
freq_q1,725.0,-2.500000e-01,4.333767e-06,-2.500150e-01,-2.500035e-01,-0.250002,-2.499965e-01,-2.499858e-01
freq_q3,725.0,2.499922e-01,5.490439e-06,2.499551e-01,2.499892e-01,0.249994,2.499965e-01,2.499989e-01
freq_mode,725.0,-4.999961e-01,4.843362e-06,-5.000000e-01,-5.000000e-01,-0.500000,-4.999930e-01,-4.999715e-01


## 3. Distribución de la Variable Objetivo (Real vs IA)
Comprobamos el balanceo de nuestras clases. `bonafide` representa audios humanos, y `spoof` representa audios generados por IA.

In [5]:
class_counts = df['Key'].value_counts().reset_index()
class_counts.columns = ['Tipo de Audio', 'Cantidad']

fig = px.pie(class_counts, names='Tipo de Audio', values='Cantidad', 
             title='Proporción de Audios Humanos (bonafide) vs IA (spoof)',
             color='Tipo de Audio',
             color_discrete_map={'spoof': '#EF553B', 'bonafide': '#00CC96'},
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 4. Análisis de Características: Humanos vs IA
A continuación visualizamos cómo se distribuyen las distintas métricas de audio dependiendo de si la muestra es real o generada.
Seleccionaremos las características más representativas como la energía, el centroide espectral, la tasa de cruces por cero (zero crossings), entre otras.

In [6]:
features_to_plot = ['energy', 'rmse_mean', 'zero_crossings', 'mfcc_mean', 
                    'spec_centroid_mean', 'spec_bandwidth_mean', 'tempo']

for feature in features_to_plot:
    fig = px.histogram(df, x=feature, color='Key', barmode='overlay', 
                       marginal='box', 
                       title=f'Distribución de {feature} por Tipo de Audio',
                       color_discrete_map={'spoof': '#EF553B', 'bonafide': '#00CC96'}, 
                       opacity=0.7)
    fig.show()

### 4.1 Observaciones de las Distribuciones:
- **Boxplots Superiores**: Los gráficos de caja (boxplots) en la parte superior de cada histograma ayudan a visualizar fácilmente la mediana, los cuartiles y la presencia de valores atípicos (outliers) en ambos grupos.
- Las variables donde las distribuciones (y por ende, las cajas) de 'spoof' y 'bonafide' están más separadas son mejores candidatas para que un modelo de Machine Learning aprenda a diferenciarlos.

## 5. Análisis de Correlación
Revisamos cómo se correlacionan linealmente las variables numéricas entre sí.

In [7]:
# Seleccionamos solo las variables numéricas para la correlación
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig = go.Figure(data=go.Heatmap(
                   z=corr_matrix.values,
                   x=corr_matrix.columns,
                   y=corr_matrix.columns,
                   colorscale='RdBu',
                   zmin=-1, zmax=1))

fig.update_layout(title='Matriz de Correlación de Características de Audio',
                  width=900, height=900)
fig.show()

## 6. Gráficos de Violín (Violin Plots)
Los violin plots son excelentes para comparar la forma completa de las distribuciones. Ayudan a ver si hay bimodalidad y dónde se concentra la mayor densidad de probabilidad, de una forma más elegante y descriptiva que un simple boxplot.

In [8]:
for feature in features_to_plot:
    fig = px.violin(df, y=feature, x="Key", color="Key", box=True, points="all",
                    title=f'Violin Plot de {feature} - Humanos vs IA',
                    color_discrete_map={'spoof': '#EF553B', 'bonafide': '#00CC96'})
    fig.show()

## 7. Gráficos de Dispersión (Scatter Matrix o Pairplot)
Para ver relaciones en 2D entre las principales características que hemos identificado.

In [9]:
key_features = ['zero_crossings', 'mfcc_mean', 'spec_centroid_mean', 'energy']

fig = px.scatter_matrix(df, dimensions=key_features, color="Key",
                        title="Scatter Matrix de Características Clave",
                        color_discrete_map={'spoof': '#EF553B', 'bonafide': '#00CC96'}, 
                        opacity=0.6)
fig.update_layout(width=1000, height=1000)
fig.show()

## Conclusiones Preliminares
A través de estos gráficos interactivos provistos por Plotly, podemos identificar qué parámetros (features) extraídos de los audios difieren más notablemente entre las muestras humanas y las muestras generadas por IA. Esas características serán las que tengan el mayor peso predictivo al momento de entrenar nuestros algoritmos de clasificación de Deep Learning o Machine Learning tradicional.